# Chẩn đoán bệnh lá cà phê — MobileNetV3Large trên JMuBEN (+ RoCoLe Healthy)
## Phương pháp
1. **Chống Data Leakage 2 tầng:**
   - **4 lớp bệnh JMuBEN (`cercospora`, `leaf_rust`, `miner`, `phoma`):** Dihedral 64-bit DCT pHash (Hamming ≤ 4) + connected components trên frozen ImageNet MobileNetV3Large embedding 8 hướng (`cosine ≥ 0.85`). Loại `JMuBEN/Healthy` vì 18,983 ảnh thực chất chỉ sinh từ 2 lá gốc.
   - **Lớp `healthy` (RoCoLe `coffee___healthy`):** pHash (Hamming ≤ 4) + mã lá vật lý `C<x>P<y>` (giữ mặt trước/sau cùng 1 lá trong cùng 1 nhóm). Ghi nhận rõ **domain shift** giữa ảnh thực địa Robusta và ảnh cận JMuBEN.
   - **Kiểm chứng chống leak tự động:** 0 giao nhóm, 0 giao canonical pHash, 0 giao mã lá RoCoLe, `max cross-split cosine < 0.85` trên JMuBEN.
2. **Đánh giá thống kê bằng 5-Fold StratifiedGroupKFold Cross-Validation:**
   - Mỗi fold giữ 1 fold làm `test`, 1 fold làm `val` (chọn checkpoint), 3 fold còn lại làm `train`.
   - `val` và `test` chỉ giữ **1 ảnh đại diện / nhóm độc lập** để mỗi nhóm có cùng trọng số khi đánh giá.
   - Báo cáo **`mean ± std` qua 5 fold** lẫn **Out-Of-Fold (OOF)** trên toàn bộ nhóm sau khi lọc.
3. **Kiểm soát overfitting khi fine-tune:**
   - Dùng `CosineAnnealingLR`, `label_smoothing = 0.05`, `weight_decay = 1e-3`, `RandomResizedCrop`, `RandomRotation(20)`, `ColorJitter`, `RandomErasing(0.15)` và early stopping theo `val macro-F1`.

Run All với GPU T4 và Internet off. Attach 3 dataset trong `kernel-metadata.json`.

## 1. Môi trường và cấu hình thực nghiệm

In [ ]:
import hashlib, json, platform, random, re, time, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.fft import dctn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

CFG = dict(
    seed=42,
    n_splits=5,
    image_size=224,
    batch_size=64,
    workers=2,
    warmup_epochs=3,
    epochs=25,
    patience=6,
    lr_head=1e-3,
    lr_finetune=1e-4,
    lr_min=1e-6,
    weight_decay=1e-3,
    label_smoothing=0.05,
    phash_distance=4,
    emb_cosine_threshold=0.85,
)
OUT = Path('/kaggle/working/results')
OUT.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), 'GPU required: enable GPU in Kaggle settings.'
DEVICE = torch.device('cuda')
torch.set_num_threads(2)

jmuben_roots = sorted(p for p in Path('/kaggle/input').rglob('JMuBEN') if p.is_dir())
assert len(jmuben_roots) == 1, f'Expected exactly one JMuBEN directory, found {jmuben_roots}'
JMUBEN_ROOT = jmuben_roots[0]

rocole_healthy_roots = sorted(p for p in Path('/kaggle/input').rglob('coffee___healthy') if p.is_dir())
assert len(rocole_healthy_roots) == 1, f'Expected exactly one RoCoLe coffee___healthy directory, found {rocole_healthy_roots}'
ROCOLE_HEALTHY_ROOT = rocole_healthy_roots[0]

SOURCE_ROOTS = {'jmuben': JMUBEN_ROOT, 'rocole': ROCOLE_HEALTHY_ROOT.parent}

def resolve_path(source, rel_path):
    return SOURCE_ROOTS[source] / rel_path

weight_paths = list(Path('/kaggle/input').rglob('mobilenet_v3_large-5c1a4163.pth'))
assert len(weight_paths) == 1, 'Attach the official ImageNet V2 weight dataset.'
WEIGHTS = weight_paths[0]
weight_sha = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
assert weight_sha.startswith('5c1a4163'), 'Pretrained weight checksum mismatch.'

environment = dict(
    python=platform.python_version(),
    torch=torch.__version__,
    torchvision=torchvision.__version__,
    gpu=torch.cuda.get_device_name(0),
    weights_sha256=weight_sha,
    jmuben_root=str(JMUBEN_ROOT),
    rocole_healthy_root=str(ROCOLE_HEALTHY_ROOT),
    config=CFG,
)
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))

## 2. Thu thập dữ liệu và kiểm toán ảnh thô (Raw Data Audit)

In [ ]:
def group_hashes(hash_rows, threshold):
    parent = list(range(len(hash_rows)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    tree = None
    for i, hashes in enumerate(hash_rows):
        for value in hashes:
            value = int(value)
            stack = [tree] if tree is not None else []
            while stack:
                node = stack.pop()
                distance = (value ^ node[0]).bit_count()
                if distance <= threshold:
                    a, b = find(i), find(node[1])
                    parent[max(a, b)] = min(a, b)
                stack.extend(child for edge, child in node[2].items()
                             if distance - threshold <= edge <= distance + threshold)
            if tree is None:
                tree = [value, i, {}]
            else:
                node = tree
                while True:
                    distance = (value ^ node[0]).bit_count()
                    if distance == 0:
                        break
                    if distance not in node[2]:
                        node[2][distance] = [value, i, {}]
                        break
                    node = node[2][distance]
    return [find(i) for i in range(len(hash_rows))]

assert group_hashes([[0], [1], [3], [255]], 1) == [0, 0, 0, 3]
assert group_hashes([[0, 255], [254], [15]], 1) == [0, 0, 2]

def extract_rocole_leaf_id(filename):
    m = re.match(r'^(C\d+P\d+)', filename, flags=re.IGNORECASE)
    if m is None:
        raise ValueError(f'Cannot read RoCoLe leaf ID from {filename!r}')
    return m.group(1).upper()

def inspect(item):
    source, path = item
    root = SOURCE_ROOTS[source]
    raw_label = path.parent.name
    if source == 'rocole':
        label = 'healthy'
        leaf_id = f'rocole_{extract_rocole_leaf_id(path.stem)}'
    else:
        norm = raw_label.strip().lower().replace('cerscospora', 'cercospora')
        label = norm.replace(' ', '_')
        leaf_id = ''
    record = dict(source=source, path=str(path.relative_to(root)), raw_label=raw_label,
                  label=label, leaf_id=leaf_id)
    try:
        with Image.open(path) as im:
            im.load()
            im = ImageOps.exif_transpose(im)
            record.update(width=im.width, height=im.height)
            gray = np.asarray(im.convert('L').resize((32, 32), Image.Resampling.BILINEAR), dtype=np.float32)
        hashes = []
        for flip in (gray, np.fliplr(gray)):
            for k in range(4):
                values = dctn(np.rot90(flip, k), type=2, norm='ortho')[:8, :8].ravel()
                bits = values > np.median(values[1:])
                bits[0] = False
                hashes.append(int.from_bytes(np.packbits(bits).tobytes(), 'big'))
        record.update(canonical=f'{min(hashes):016x}', hashes=hashes, error='')
    except (OSError, ValueError) as exc:
        record.update(error=str(exc))
    return record

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
jmuben_items = [('jmuben', p) for p in sorted(JMUBEN_ROOT.rglob('*'))
                if p.suffix.lower() in IMG_EXTS and '__MACOSX' not in p.parts]
rocole_items = [('rocole', p) for p in sorted(ROCOLE_HEALTHY_ROOT.rglob('*'))
                if p.suffix.lower() in IMG_EXTS and '__MACOSX' not in p.parts]
items = jmuben_items + rocole_items
assert jmuben_items and rocole_items, f'Missing images: jmuben={len(jmuben_items)}, rocole={len(rocole_items)}'

with ThreadPoolExecutor(4) as pool:
    records = list(pool.map(inspect, items))
raw = pd.DataFrame(records)
raw.drop(columns=['hashes'], errors='ignore').to_csv(OUT / 'raw_manifest.csv', index=False)
valid = raw[raw.error == ''].copy().reset_index(drop=True)
assert valid.label.nunique() == 5, f'Expected 5 labels: {valid.label.value_counts()}'
raw_counts = valid.label.value_counts().sort_index()
print('Raw counts by source and label:')
display(pd.crosstab(valid.label, valid.source, margins=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw_counts.plot.bar(ax=axes[0], rot=20, title='Raw class distribution (JMuBEN + RoCoLe healthy)')
axes[1].scatter(valid.width, valid.height, alpha=.2, s=6)
axes[1].set(xlabel='Width', ylabel='Height', title='Raw image dimensions')
plt.tight_layout(); plt.savefig(OUT / 'raw_eda.png'); plt.show()

fig, axes = plt.subplots(5, 4, figsize=(10, 12))
for row_axes, (label, frame) in zip(axes, valid.groupby('label', sort=True)):
    for ax, (_, row) in zip(row_axes, frame.sample(min(4, len(frame)), random_state=42).iterrows()):
        with Image.open(resolve_path(row.source, row.path)) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(f'{label} ({row.source})', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'raw_samples.png'); plt.show()

## 3. Gộp nhóm chống Data Leakage 2 tầng (pHash + Frozen CNN Embedding + RoCoLe Leaf ID)

In [ ]:
jmuben_healthy_raw = int(((valid.source == 'jmuben') & (valid.label == 'healthy')).sum())
print(f'Excluding JMuBEN/Healthy ({jmuben_healthy_raw:,} raw images -> only 2 unique original leaves); using RoCoLe coffee___healthy instead.')
valid = valid[~((valid.source == 'jmuben') & (valid.label == 'healthy'))].copy().reset_index(drop=True)

print('Stage 1: pHash grouping on', len(valid), 'valid images...', flush=True)
valid['phash_group'] = group_hashes(valid.hashes, CFG['phash_distance'])
valid['duplicate_phash'] = valid.duplicated('canonical')

cand = valid[~valid.duplicate_phash].copy().reset_index(drop=True)
print(f'After canonical pHash dedup: {len(cand):,} / {len(valid):,} images retained.')
display(pd.crosstab(cand.label, cand.source, margins=True))

print('Stage 2: Extracting frozen ImageNet MobileNetV3Large 8-view embeddings on GPU...', flush=True)
eval_transform = MobileNet_V3_Large_Weights.IMAGENET1K_V2.transforms()

class PathDataset(Dataset):
    def __init__(self, frame):
        self.sources = frame.source.tolist()
        self.paths = frame.path.tolist()
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(resolve_path(self.sources[i], self.paths[i])) as im:
            return eval_transform(ImageOps.exif_transpose(im).convert('RGB'))

frozen = mobilenet_v3_large(weights=None)
frozen.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))
frozen = frozen.to(DEVICE).eval()

emb_loader = DataLoader(PathDataset(cand), batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['workers'], pin_memory=True)
emb_chunks = []
with torch.inference_mode():
    for x in emb_loader:
        x = x.to(DEVICE, non_blocking=True)
        views = [torch.rot90(v, k, (2, 3)) for v in (x, x.flip(3)) for k in range(4)]
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feat = torch.stack([frozen.avgpool(frozen.features(v)).flatten(1) for v in views]).mean(0)
        feat = nn.functional.normalize(feat.float(), dim=1)
        emb_chunks.append(feat.cpu().numpy())
del frozen
torch.cuda.empty_cache()

E = np.concatenate(emb_chunks, axis=0)
C = E @ E.T
np.fill_diagonal(C, -1.0)

is_jmuben = (cand.source == 'jmuben').to_numpy()
adj = is_jmuben[:, None] & is_jmuben[None, :] & (C >= CFG['emb_cosine_threshold'])

def add_group_edges(adjacency, full_rows, canonical_hashes):
    # Duplicate images still link their canonical representative to the other photos of the leaf.
    canonical_index = pd.Series(np.arange(len(canonical_hashes)), index=canonical_hashes)
    nodes = full_rows.canonical.map(canonical_index)
    for key in ('phash_group', 'leaf_id'):
        for group_key, members in nodes.groupby(full_rows[key], sort=False):
            if group_key == '':
                continue
            indices = members.unique()
            adjacency[indices[0], indices] = True
            adjacency[indices, indices[0]] = True

add_group_edges(adj, valid, cand.canonical)

n_groups, comp = connected_components(csr_matrix(adj), directed=False)
cand['group'] = comp

canon_to_group = dict(zip(cand.canonical, cand.group))
valid['group'] = valid.canonical.map(canon_to_group)

conflicts = valid.groupby('group').label.nunique()
bad_groups = set(conflicts[conflicts > 1].index)
cand['excluded_conflict'] = cand.group.isin(bad_groups)
valid['excluded_conflict'] = valid.group.isin(bad_groups)
valid.drop(columns=['hashes']).to_csv(OUT / 'group_audit.csv', index=False)

print(f'Total independent groups (Stage 2, tau={CFG["emb_cosine_threshold"]}): {n_groups}; multi-label conflict groups: {len(bad_groups)}')
display(cand[~cand.excluded_conflict].groupby(['label', 'source']).group.nunique().unstack(fill_value=0))

## 4. Gán 5-Fold StratifiedGroupKFold và Kiểm chứng Chống Data Leakage trên mọi Fold

- Gán `fold ∈ {0, 1, 2, 3, 4}` cho từng nhóm độc lập bằng `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`.
- Kiểm chứng trên **cả 5 fold**: 0 giao nhóm, 0 giao canonical pHash, 0 giao mã lá RoCoLe, và `max cross-fold cosine < 0.85` trên các ảnh JMuBEN giữ lại sau khi lọc pHash.

In [ ]:
keep_mask = (~cand.excluded_conflict).to_numpy()
data_all = cand[keep_mask].copy().reset_index(drop=True)
C_clean = C if keep_mask.all() else C[np.ix_(keep_mask, keep_mask)]
del C, E, emb_chunks, adj

classes = sorted(data_all.label.unique())
assert len(classes) == 5, f'Filtering removed a class: {classes}'
class_to_idx = {c: i for i, c in enumerate(classes)}
data_all['target'] = data_all.label.map(class_to_idx)
assert data_all.groupby('label').group.nunique().min() >= CFG['n_splits'], 'Need >=5 independent groups per class.'

splitter = StratifiedGroupKFold(n_splits=CFG['n_splits'], shuffle=True, random_state=CFG['seed'])
data_all['fold'] = -1
for fold, (_, idx) in enumerate(splitter.split(data_all, data_all.target, data_all.group)):
    data_all.loc[idx, 'fold'] = fold
assert (data_all.fold >= 0).all()

# Strict cross-fold leakage assertions
assert data_all.groupby('group').fold.nunique().max() == 1, 'Group leaked across folds!'
assert data_all.groupby('canonical').fold.nunique().max() == 1, 'Canonical pHash leaked across folds!'
# Check the full manifest, including photos dropped by canonical deduplication.
full_split = valid.loc[~valid.excluded_conflict, ['canonical', 'phash_group', 'leaf_id', 'group']].copy()
full_split['fold'] = full_split.group.map(data_all.groupby('group').fold.first())
assert full_split.fold.notna().all(), 'Missing fold for a retained group!'
assert full_split.groupby('phash_group').fold.nunique().max() == 1, 'pHash group leaked across folds!'
rocole_only = full_split[full_split.leaf_id != '']
assert rocole_only.groupby('leaf_id').fold.nunique().max() == 1, 'RoCoLe leaf_id leaked across folds!'
for f in range(CFG['n_splits']):
    assert set(data_all[data_all.fold == f].label) == set(classes), f'Missing class in fold {f}!'

# Group representatives table (1 rep per independent group, used for unbiased val/test & OOF)
rep_mask = (~data_all.duplicated('group')).to_numpy()
reps = data_all[rep_mask].copy().reset_index(drop=True)
# Cross-fold JMuBEN cosine audit across all 5 folds
jm_all = (data_all.source == 'jmuben').to_numpy()
fold_arr = data_all.fold.to_numpy()
y_all = data_all.target.to_numpy()
max_cross_cos = -1.0
nn_agrees = []
for f in range(CFG['n_splits']):
    ia = np.flatnonzero((fold_arr == f) & jm_all)
    ib = np.flatnonzero((fold_arr != f) & jm_all)
    sub = C_clean[np.ix_(ia, ib)]
    max_cross_cos = max(max_cross_cos, float(sub.max()))
    rep_rows = ia[rep_mask[ia]]
    rep_nearest = sub[rep_mask[ia]].argmax(1)
    nn_agrees.append(float((y_all[ib[rep_nearest]] == y_all[rep_rows]).mean()))
assert max_cross_cos < CFG['emb_cosine_threshold'], f'Cross-fold JMuBEN cosine leak: {max_cross_cos}'

# Visual audit: top-6 closest JMuBEN cross-fold pairs (fold 0 test vs train folds 2,3,4)
te_idx = np.where((fold_arr == 0) & jm_all & rep_mask)[0]
tr_idx = np.flatnonzero((fold_arr >= 2) & jm_all)
sub_tt = C_clean[np.ix_(te_idx, tr_idx)]
best_tt = sub_tt.max(1)
nn_tt = sub_tt.argmax(1)
order = np.argsort(-best_tt)[:6]
fig, axes = plt.subplots(len(order), 2, figsize=(6, 2.6 * len(order)), squeeze=False)
for r, o in enumerate(order):
    for c, idx_img in enumerate((te_idx[o], tr_idx[nn_tt[o]])):
        row = data_all.iloc[idx_img]
        with Image.open(resolve_path(row.source, row.path)) as im:
            axes[r, c].imshow(ImageOps.exif_transpose(im).convert('RGB'))
        role = 'fold0-test' if c == 0 else f'fold{row.fold}-train'
        axes[r, c].set_title(f"{role} | {row.label} | cos={best_tt[o]:.3f}", fontsize=8)
        axes[r, c].axis('off')
plt.tight_layout(); plt.savefig(OUT / 'cross_split_nearest_pairs.png'); plt.show()

del C_clean

# Save manifests and audit
data_all['is_group_rep'] = rep_mask
data_all.drop(columns=['hashes'], errors='ignore').to_csv(OUT / 'split_manifest.csv', index=False)

fold_rep_counts = pd.crosstab(reps.label, reps.fold)
display(fold_rep_counts)
fold_rep_counts.plot.bar(rot=15, figsize=(9, 4))
plt.ylabel('Independent groups (1 rep/group)'); plt.title('Independent groups per class across 5 folds')
plt.tight_layout(); plt.savefig(OUT / 'class_distribution.png'); plt.show()

comparison = pd.DataFrame({'raw': raw_counts, 'phash_retained': data_all.label.value_counts().sort_index(),
                           'independent_groups': reps.label.value_counts().sort_index()})
comparison['group_percent_of_raw'] = 100 * comparison.independent_groups / comparison.raw
comparison.to_csv(OUT / 'before_after.csv')
display(comparison)

leak_stats = dict(
    jmuben_all5folds_max_cross_cos=max_cross_cos,
    jmuben_all5folds_mean_frozen_1nn_agreement=round(float(np.mean(nn_agrees)), 4),
    jmuben_fold0_test_to_train_max_cos=round(float(best_tt.max()), 4),
    jmuben_fold0_test_to_train_median_cos=round(float(np.median(best_tt)), 4),
)
audit = dict(
    raw_count=len(raw),
    corrupt_count=int((raw.error != '').sum()),
    phash_dedup_count=len(cand),
    conflict_images=int(valid.excluded_conflict.sum()),
    conflict_groups=len(bad_groups),
    independent_groups=int(reps.group.nunique()),
    n_splits=CFG['n_splits'],
    phash_threshold=CFG['phash_distance'],
    emb_cosine_threshold=CFG['emb_cosine_threshold'],
    val_test_one_rep_per_group=True,
    rocole_leaf_ids_audited=int(rocole_only.leaf_id.nunique()),
    fold_group_counts=fold_rep_counts.to_dict(),
    source_group_counts=pd.crosstab(reps.label, reps.source).to_dict(),
    cross_fold_leakage_audit=leak_stats,
    domain_shift_note='JMuBEN Healthy had only 2 unique original leaves; RoCoLe coffee___healthy was used for healthy (grouped by C<x>P<y> leaf_id), introducing Robusta field-background domain shift.',
)
(OUT / 'data_audit.json').write_text(json.dumps(audit, indent=2))
(OUT / 'classes.json').write_text(json.dumps(class_to_idx, indent=2))
print(json.dumps(audit, indent=2))

decision_matrix = pd.DataFrame([
    ['Nhãn thô không đồng nhất', 'Tách sai lớp', 'Chuẩn hóa snake_case, sửa Cerscospora', 'raw_manifest.csv'],
    ['Ít lá Healthy gốc trong JMuBEN', 'Khó tạo đủ nhóm lá cho cả ba tập', 'Thay bằng RoCoLe coffee___healthy (gộp mã lá C<x>P<y>), ghi rõ domain shift', 'data_audit.json'],
    ['Ảnh gần trùng do dịch chuyển, thu phóng hoặc làm mờ', 'pHash có thể bỏ sót ảnh gần trùng', 'Gộp 2 tầng: pHash + frozen CNN cosine >= 0.85', 'group_audit.csv'],
    ['Ít nhóm cercospora', 'Kết quả từ một lần chia dễ dao động', '5-Fold StratifiedGroupKFold CV (mean ± std + OOF toàn bộ nhóm)', 'cv_fold_results.csv'],
    ['Số nhóm lá dùng để học còn hạn chế', 'Mô hình dễ quá khớp', 'CosineAnnealingLR + label_smoothing=0.05 + weight_decay=1e-3 + RandomErasing', 'learning_curves.png'],
], columns=['Phát hiện', 'Rủi ro', 'Quyết định', 'Kiểm chứng'])
display(decision_matrix)
decision_matrix.to_csv(OUT / 'preprocessing_decisions.csv', index=False)

## 5. Huấn luyện 5-Fold Cross-Validation (Warm-up + Fine-tune + CosineAnnealingLR + Label Smoothing)

Mỗi fold `k ∈ {0..4}`:
- `test` = fold `k` (1 đại diện / nhóm độc lập)
- `val` = fold `(k + 1) % 5` (1 đại diện / nhóm độc lập, dùng chọn checkpoint & early stopping)
- `train` = 3 fold còn lại (toàn bộ ảnh canonical pHash thuộc 3 fold này)
- Lưu `best.pt` của Fold 0 để xuất Grad-CAM, suy luận CPU độc lập và TorchScript.

In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(0.15, 0.15, 0.15, 0.03),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.15, scale=(0.02, 0.12)),
])

class Leaves(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(resolve_path(row.source, row.path)) as im:
            image = self.transform(ImageOps.exif_transpose(im).convert('RGB'))
        return image, int(row.target)

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed); random.seed(seed)

def make_loader(frame, transform, shuffle, seed):
    return DataLoader(
        Leaves(frame, transform),
        batch_size=CFG['batch_size'],
        shuffle=shuffle,
        num_workers=CFG['workers'],
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
    )

def run_epoch(model, loader, criterion, class_weights, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    if training:
        for module in model.features.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
    total_loss, total_weight = 0.0, 0.0
    labels, probabilities = [], []
    with torch.set_grad_enabled(training):
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, targets)
            assert torch.isfinite(loss), 'Non-finite loss.'
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer); scaler.update()
            batch_weight = class_weights[targets].sum().item()
            total_loss += loss.item() * batch_weight
            total_weight += batch_weight
            labels.extend(targets.detach().cpu().tolist())
            probabilities.append(logits.detach().float().softmax(1).cpu().numpy())
    probs = np.concatenate(probabilities)
    pred = probs.argmax(1)
    return dict(
        loss=total_loss / total_weight,
        accuracy=float(accuracy_score(labels, pred)),
        macro_f1=float(f1_score(labels, pred, labels=list(range(len(classes))), average='macro', zero_division=0)),
    ), np.array(labels), probs

disease_ids = [class_to_idx[c] for c in classes if c != 'healthy']
fold_results = []
oof_frames = []
all_histories = []
fold0_generalization = None
fold0_probs = None

total_started = time.perf_counter()
for test_fold in range(CFG['n_splits']):
    val_fold = (test_fold + 1) % CFG['n_splits']
    train_folds = [f for f in range(CFG['n_splits']) if f not in (test_fold, val_fold)]
    set_seed(CFG['seed'] + test_fold)

    train_df = data_all[data_all.fold.isin(train_folds)].reset_index(drop=True)
    val_df = reps[reps.fold == val_fold].reset_index(drop=True)
    test_df = reps[reps.fold == test_fold].reset_index(drop=True)

    train_loader = make_loader(train_df, train_transform, True, CFG['seed'] + test_fold)
    val_loader = make_loader(val_df, eval_transform, False, CFG['seed'] + test_fold)
    test_loader = make_loader(test_df, eval_transform, False, CFG['seed'] + test_fold)

    model = mobilenet_v3_large(weights=None)
    model.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(classes))
    model = model.to(DEVICE)
    for param in model.features.parameters():
        param.requires_grad = False

    counts = np.bincount(train_df.target.to_numpy(), minlength=len(classes))
    class_weights = torch.tensor(len(train_df) / (len(classes) * counts), dtype=torch.float32, device=DEVICE)
    if test_fold == 0:
        pd.DataFrame({'train_count': counts, 'loss_weight': class_weights.cpu().numpy()}, index=classes).to_csv(OUT / 'class_weights.csv')

    train_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CFG['label_smoothing'])
    eval_criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])
    scheduler = None
    scaler = torch.amp.GradScaler('cuda')

    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    f_start = time.perf_counter()
    for epoch in range(CFG['epochs']):
        if epoch == CFG['warmup_epochs']:
            for param in model.features.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr_finetune'],
                                          weight_decay=CFG['weight_decay'])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=CFG['epochs'] - CFG['warmup_epochs'], eta_min=CFG['lr_min']
            )
            stale = 0
        tr_m, _, _ = run_epoch(model, train_loader, train_criterion, class_weights, optimizer, scaler)
        va_m, _, _ = run_epoch(model, val_loader, eval_criterion, class_weights)
        row = dict(
            fold=test_fold, epoch=epoch + 1,
            stage='warmup' if epoch < CFG['warmup_epochs'] else 'finetune',
            lr=optimizer.param_groups[0]['lr'],
            **{f'train_{k}': v for k, v in tr_m.items()},
            **{f'val_{k}': v for k, v in va_m.items()},
        )
        all_histories.append(row)
        if scheduler is not None:
            scheduler.step()
        if va_m['macro_f1'] > best_f1:
            best_f1, best_epoch, stale = va_m['macro_f1'], epoch + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        if epoch >= CFG['warmup_epochs'] and stale >= CFG['patience']:
            break

    f_sec = time.perf_counter() - f_start
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    te_m, truth, probs = run_epoch(model, test_loader, eval_criterion, class_weights)
    pred = probs.argmax(1)

    d_mask = np.array([classes[i] != 'healthy' for i in truth])
    d_acc = float(accuracy_score(truth[d_mask], pred[d_mask]))
    d_f1 = float(f1_score(truth[d_mask], pred[d_mask], labels=disease_ids, average='macro', zero_division=0))
    w_f1 = float(f1_score(truth, pred, average='weighted'))

    fold_rec = dict(
        fold=test_fold, val_fold=val_fold, best_epoch=best_epoch, seconds=round(f_sec, 2),
        val_macro_f1=best_f1, test_loss=te_m['loss'], test_accuracy=te_m['accuracy'],
        test_macro_f1=te_m['macro_f1'], test_weighted_f1=w_f1,
        jmuben_4disease_accuracy=d_acc, jmuben_4disease_macro_f1=d_f1,
        test_count=len(truth),
    )
    fold_results.append(fold_rec)
    print(f'Fold {test_fold} done ({f_sec:.1f}s):', json.dumps(fold_rec), flush=True)

    # Save Fold 0 checkpoint & clean-train generalization for deployment & Grad-CAM
    if test_fold == 0:
        torch.save(dict(model=best_state, fold=0, epoch=best_epoch, val_macro_f1=best_f1,
                        classes=classes, config=CFG, architecture='mobilenet_v3_large',
                        pretrained='IMAGENET1K_V2'), OUT / 'best.pt')
        clean_tr_loader = make_loader(train_df, eval_transform, False, CFG['seed'])
        tr_clean_m, _, _ = run_epoch(model, clean_tr_loader, eval_criterion, class_weights)
        va_best_m, _, _ = run_epoch(model, val_loader, eval_criterion, class_weights)
        fold0_generalization = pd.DataFrame([tr_clean_m, va_best_m, te_m], index=['train_clean', 'val', 'test'])
        fold0_generalization.to_csv(OUT / 'generalization.csv')
        fold0_probs = probs

    tf_df = test_df[['source', 'path', 'label', 'group', 'fold']].copy()
    tf_df['target'] = truth
    tf_df['pred_idx'] = pred
    tf_df['prediction'] = [classes[i] for i in pred]
    for i, cls in enumerate(classes):
        tf_df[f'prob_{cls}'] = probs[:, i]
    tf_df['confidence'] = probs.max(1)
    oof_frames.append(tf_df)

training_seconds = time.perf_counter() - total_started
cv_df = pd.DataFrame(fold_results)
cv_df.to_csv(OUT / 'cv_fold_results.csv', index=False)
hist_df = pd.DataFrame(all_histories)
hist_df.to_csv(OUT / 'history.csv', index=False)
oof_df = pd.concat(oof_frames, ignore_index=True)
assert oof_df.group.is_unique and set(oof_df.group) == set(reps.group), 'OOF must predict each group once.'
oof_df.to_csv(OUT / 'test_predictions.csv', index=False)
print(f'All 5 folds complete in {training_seconds/60:.2f} min.')
display(cv_df)

## 6. Tổng hợp 5-Fold CV (`mean ± std`), Out-Of-Fold (OOF) và Trực quan hóa

In [ ]:
truth_oof = oof_df.target.to_numpy()
pred_oof = oof_df.pred_idx.to_numpy()
d_mask_oof = np.array([classes[i] != 'healthy' for i in truth_oof])

oof_report = classification_report(truth_oof, pred_oof, labels=list(range(len(classes))),
                                   target_names=classes, output_dict=True, zero_division=0)
pd.DataFrame(oof_report).transpose().to_csv(OUT / 'classification_report.csv')
display(pd.DataFrame(oof_report).transpose())

cv_summary = dict(
    n_splits=CFG['n_splits'],
    oof_total_groups=len(oof_df),
    cv_accuracy_mean=float(cv_df.test_accuracy.mean()),
    cv_accuracy_std=float(cv_df.test_accuracy.std(ddof=1)),
    cv_macro_f1_mean=float(cv_df.test_macro_f1.mean()),
    cv_macro_f1_std=float(cv_df.test_macro_f1.std(ddof=1)),
    cv_weighted_f1_mean=float(cv_df.test_weighted_f1.mean()),
    cv_weighted_f1_std=float(cv_df.test_weighted_f1.std(ddof=1)),
    cv_jmuben_4disease_accuracy_mean=float(cv_df.jmuben_4disease_accuracy.mean()),
    cv_jmuben_4disease_accuracy_std=float(cv_df.jmuben_4disease_accuracy.std(ddof=1)),
    cv_jmuben_4disease_macro_f1_mean=float(cv_df.jmuben_4disease_macro_f1.mean()),
    cv_jmuben_4disease_macro_f1_std=float(cv_df.jmuben_4disease_macro_f1.std(ddof=1)),
    oof_accuracy=float(accuracy_score(truth_oof, pred_oof)),
    oof_macro_f1=float(f1_score(truth_oof, pred_oof, labels=list(range(len(classes))), average='macro', zero_division=0)),
    oof_weighted_f1=float(f1_score(truth_oof, pred_oof, average='weighted')),
    oof_jmuben_4disease_accuracy=float(accuracy_score(truth_oof[d_mask_oof], pred_oof[d_mask_oof])),
    oof_jmuben_4disease_macro_f1=float(f1_score(truth_oof[d_mask_oof], pred_oof[d_mask_oof], labels=disease_ids, average='macro', zero_division=0)),
    training_seconds=training_seconds,
    parameters=sum(p.numel() for p in model.parameters()),
    seed=CFG['seed'],
)
(OUT / 'metrics.json').write_text(json.dumps(cv_summary, indent=2))
print(json.dumps(cv_summary, indent=2))

# Pooled OOF confusion matrix across ALL independent groups
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(truth_oof, pred_oof, labels=list(range(len(classes))),
    display_labels=classes, ax=ax, cmap='Blues', xticks_rotation=30, colorbar=False)
ax.set_title(f'5-Fold Out-Of-Fold Confusion Matrix ({len(oof_df)} independent groups)')
plt.tight_layout(); plt.savefig(OUT / 'confusion_matrix.png'); plt.show()

# Average only epochs completed by all five folds.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['loss', 'macro_f1']):
    for f in range(CFG['n_splits']):
        sub = hist_df[hist_df.fold == f]
        ax.plot(sub.epoch, sub[f'val_{metric}'], alpha=0.35, color='tab:orange', lw=1)
        ax.plot(sub.epoch, sub[f'train_{metric}'], alpha=0.25, color='tab:blue', lw=1)
    common_epochs = hist_df[hist_df.epoch <= hist_df.groupby('fold').epoch.max().min()]
    mean_h = common_epochs.groupby('epoch')[[f'train_{metric}', f'val_{metric}']].mean()
    ax.plot(mean_h.index, mean_h[f'train_{metric}'], color='tab:blue', lw=2.2, label='train (5-fold mean)')
    ax.plot(mean_h.index, mean_h[f'val_{metric}'], color='tab:orange', lw=2.2, label='val (5-fold mean)')
    ax.set(xlabel='Epoch', ylabel=metric, title=f'5-Fold Learning Curve ({metric})')
    ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUT / 'learning_curves.png'); plt.show()

# Generalization bar chart (Fold 0)
fold0_generalization[['accuracy', 'macro_f1']].plot.bar(ylim=(0, 1), rot=0, figsize=(8, 4), title='Fold 0 Generalization')
plt.tight_layout(); plt.savefig(OUT / 'generalization.png'); plt.show()

# Top misclassified OOF images
errors = oof_df[oof_df.label != oof_df.prediction].sort_values('confidence', ascending=False).head(12)
if len(errors):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, row) in zip(axes.ravel(), errors.iterrows()):
        with Image.open(resolve_path(row.source, row.path)) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(f'F{row.fold}: {row.label} → {row.prediction} ({row.confidence:.2f})', fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / 'misclassified.png'); plt.show()

## 7. Grad-CAM, Đóng gói TorchScript và Kiểm thử Suy luận Độc lập (Fold 0)

In [ ]:
checkpoint = torch.load(OUT / 'best.pt', map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint['model'])
model.eval()

def grad_cam(image):
    activation = []
    handle = model.features[-1].register_forward_hook(lambda module, inputs, output: activation.append(output))
    try:
        model.eval()
        x = eval_transform(image).unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = model(x)
        cls = int(logits.argmax(1).item())
        gradient = torch.autograd.grad(logits[0, cls], activation[0])[0]
        cam = (gradient.mean((2, 3), keepdim=True) * activation[0]).sum(1, keepdim=True).relu()
        cam = nn.functional.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        rgb = x[0].detach().cpu() * torch.tensor([.229, .224, .225])[:, None, None] + torch.tensor([.485, .456, .406])[:, None, None]
        return rgb.permute(1, 2, 0).clamp(0, 1).numpy(), cam.detach().cpu().numpy(), classes[cls]
    finally:
        handle.remove()

fold0_oof = oof_df[oof_df.fold == 0].reset_index(drop=True)
examples = pd.concat([
    fold0_oof[fold0_oof.label == fold0_oof.prediction].groupby('label', group_keys=False).head(1).head(3),
    fold0_oof[fold0_oof.label != fold0_oof.prediction].head(3),
])
fig, axes = plt.subplots(len(examples), 2, figsize=(7, 3 * len(examples)), squeeze=False)
for row_axes, (_, row) in zip(axes, examples.iterrows()):
    with Image.open(resolve_path(row.source, row.path)) as im:
        rgb, heat, predicted = grad_cam(ImageOps.exif_transpose(im).convert('RGB'))
    row_axes[0].imshow(rgb); row_axes[1].imshow(rgb); row_axes[1].imshow(heat, cmap='jet', alpha=.4)
    row_axes[0].set_title(f'True: {row.label} ({row.source})', fontsize=9)
    row_axes[1].set_title(f'Pred: {predicted}', fontsize=9)
    for ax in row_axes: ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'gradcam.png'); plt.show()

deploy = OUT / 'deploy'
deploy.mkdir(exist_ok=True)
metadata = dict(
    architecture='mobilenet_v3_large', classes=classes, resize=232, crop=224,
    mean=[.485, .456, .406], std=[.229, .224, .225],
    weights='best.pt (fold 0)', input='RGB with EXIF orientation corrected',
    interpolation='bilinear', antialias=True,
)
(deploy / 'metadata.json').write_text(json.dumps(metadata, indent=2))

def predict_leaf_image(image_path, checkpoint_path):
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    net = mobilenet_v3_large(weights=None)
    net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, len(saved['classes']))
    net.load_state_dict(saved['model']); net.eval()
    transform = MobileNet_V3_Large_Weights.IMAGENET1K_V2.transforms()
    with Image.open(image_path) as im:
        x = transform(ImageOps.exif_transpose(im).convert('RGB')).unsqueeze(0)
    with torch.inference_mode():
        probability = net(x).softmax(1)[0]
    index = int(probability.argmax())
    return dict(label=saved['classes'][index], confidence=float(probability[index]),
                probabilities=probability.tolist())

sample = fold0_oof.iloc[0]
inference = predict_leaf_image(resolve_path(sample.source, sample.path), OUT / 'best.pt')
assert inference['label'] == sample.prediction
np.testing.assert_allclose(inference['probabilities'], fold0_probs[0], atol=3e-3, rtol=3e-3)

cpu_model = mobilenet_v3_large(weights=None)
cpu_model.classifier[-1] = nn.Linear(cpu_model.classifier[-1].in_features, len(classes))
cpu_model.load_state_dict({k: v.cpu() for k, v in checkpoint['model'].items()}); cpu_model.eval()
example = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    scripted = torch.jit.trace(cpu_model, example)
    scripted.save(str(deploy / 'mobilenetv3large.ts'))
    reloaded = torch.jit.load(str(deploy / 'mobilenetv3large.ts'))
    with Image.open(resolve_path(sample.source, sample.path)) as im:
        test_tensor = eval_transform(ImageOps.exif_transpose(im).convert('RGB')).unsqueeze(0)
    torch.testing.assert_close(reloaded(test_tensor), cpu_model(test_tensor))
print('Independent CPU inference and TorchScript round-trip passed:', inference)
(deploy / 'inference_check.json').write_text(json.dumps(inference, indent=2))

In [ ]:
per_class = pd.DataFrame(oof_report).T.loc[classes]
weakest = per_class['f1-score'].idxmin()
cm = confusion_matrix(truth_oof, pred_oof, labels=list(range(len(classes))))
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
confusion_note = (f'Nhầm nhiều nhất (OOF): {classes[i]} → {classes[j]} ({cm[i, j]} ảnh).'
                  if cm.max() else 'Không có dự đoán sai trên OOF.')

conclusions = [
    '# Kết quả MobileNetV3Large trên JMuBEN (+ RoCoLe Healthy) — 5-Fold CV theo nhóm', '',
    f'- Dữ liệu thô: {len(raw):,} ảnh (JMuBEN + RoCoLe coffee___healthy) → {len(cand):,} sau pHash → {reps.group.nunique():,} nhóm.',
    f'- **5-Fold CV (5 lớp, mean ± std):** Accuracy `{cv_summary["cv_accuracy_mean"]:.4f} ± {cv_summary["cv_accuracy_std"]:.4f}` | Macro-F1 `{cv_summary["cv_macro_f1_mean"]:.4f} ± {cv_summary["cv_macro_f1_std"]:.4f}` | Weighted-F1 `{cv_summary["cv_weighted_f1_mean"]:.4f} ± {cv_summary["cv_weighted_f1_std"]:.4f}`.',
    f'- **5-Fold CV (ảnh bệnh JMuBEN, mô hình 5 lớp):** Accuracy `{cv_summary["cv_jmuben_4disease_accuracy_mean"]:.4f} ± {cv_summary["cv_jmuben_4disease_accuracy_std"]:.4f}` | Macro-F1 `{cv_summary["cv_jmuben_4disease_macro_f1_mean"]:.4f} ± {cv_summary["cv_jmuben_4disease_macro_f1_std"]:.4f}`.',
    f'- **Out-Of-Fold (OOF, toàn bộ {len(oof_df)} nhóm):** Accuracy `{cv_summary["oof_accuracy"]:.4f}` | Macro-F1 `{cv_summary["oof_macro_f1"]:.4f}` | Macro-F1 trên ảnh bệnh JMuBEN `{cv_summary["oof_jmuben_4disease_macro_f1"]:.4f}`.',
    f'- Kiểm chứng chống leak xuyên fold JMuBEN: max cosine = `{leak_stats["jmuben_all5folds_max_cross_cos"]:.4f}` (< {CFG["emb_cosine_threshold"]}), mean frozen 1-NN = `{leak_stats["jmuben_all5folds_mean_frozen_1nn_agreement"]:.4f}`.',
    f'- Thời gian huấn luyện 5 fold: {training_seconds/60:.2f} phút trên {environment["gpu"]}.',
    f'- Lớp OOF F1 thấp nhất: `{weakest}` (`{per_class.loc[weakest, "f1-score"]:.4f}`, support={int(per_class.loc[weakest, "support"])}).',
    f'- {confusion_note}',
    '- Suy luận CPU độc lập và TorchScript (Fold 0) đã kiểm tra khớp đầu ra.', '',
    '## Hạn chế và lưu ý khoa học',
    '- **Domain shift lớp `healthy`:** `JMuBEN/Healthy` chỉ có 2 lá gốc nên thay bằng RoCoLe `coffee___healthy` (ảnh thực địa Robusta). Cần trình bày song song kết quả 4 lớp bệnh thuần JMuBEN.',
    '- Kiểm tra nhóm dùng pHash, embedding và mã lá RoCoLe. Nguồn gốc lá JMuBEN được suy ra từ ảnh. Các ngưỡng tương đồng không xác nhận được danh tính của mọi lá.',
    '- Chưa đo latency trên thiết bị di động và chưa đánh giá trên ảnh bệnh ngoài thực địa.',
]
conclusion_text = '\n'.join(conclusions)
(OUT / 'RESULTS.md').write_text(conclusion_text, encoding='utf-8')
print(conclusion_text)

with zipfile.ZipFile('/kaggle/working/report_assets.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in OUT.rglob('*'):
        if file.is_file() and file.suffix not in {'.pt', '.ts'}:
            archive.write(file, file.relative_to(OUT))
(OUT / 'SUCCESS.json').write_text(json.dumps(dict(
    status='complete', pipeline='grouped-5fold', metrics=cv_summary,
    leakage_audit=leak_stats, inference_verified=True, torchscript_verified=True,
), indent=2))
print('Grouped five-fold evaluation complete.')